In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [8]:
df = pd.read_csv("https://drive.google.com/uc?export=download&id=1EkAm2nhdqT0_1eZz1MVYP4ucI5KlXzt7", index_col=0)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   YearsExperience  30 non-null     float64
 1   Salary           30 non-null     float64
dtypes: float64(2)
memory usage: 612.0 bytes


In [9]:
# preprocessing

X = df["YearsExperience"]
Y = df["Salary"]
X = (X - np.mean(X)) / np.std(X)
Y = (Y - np.mean(Y)) / np.std(Y)

In [10]:
# Least squares

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.4, random_state = 7, shuffle = True)
sum_y = y_train.sum()
sum_x = x_train.sum()
sum_x2 = (x_train*x_train).sum()
sum_xy = (x_train*y_train).sum()
x_train, x_test, y_train, y_test = x_train.reset_index(drop=True), x_test.reset_index(drop=True), y_train.reset_index(drop=True), y_test.reset_index(drop=True)
A = np.array([[len(x_train), sum_x], [sum_x, sum_x2]])
b = np.array([[sum_y], [sum_xy]])

sol = np.linalg.solve(A, b)
m = sol[1][0]
c = sol[0][0]

y_pred = m*x_test + c
df2 = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})
ss_res = ((y_test - y_pred)**2).sum()
ss_tot = ((y_test - y_test.mean())**2).sum()

r2 = 1 - ss_res/ss_tot
print(f"y = {m:.4f}x + {c:.4f}")
print(f"r2 score = {r2}")
df2

y = 0.9979x + 0.0294
r2 score = 0.956648445382114


,Actual,Predicted
0,-1.419919,-1.334441
1,0.262859,0.024627
2,-1.105527,-1.405971
3,-0.698013,-0.547612
4,-0.749769,-0.440317
5,-0.718307,-0.833732
6,0.198860,0.239216
7,1.240203,1.240635
8,1.359074,1.562519
9,1.701773,1.884404


In [11]:
# Gradient descent

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.4, random_state = 7, shuffle = True)
m, c = 0, 0
r_target, rel_err = 0.005, 1
n = len(x_train)
epoch = 0

while rel_err > 1e-8:
    epoch += 1
    y_pred = m * x_train + c
    res = y_pred - y_train
    n = len(x_train)

    dm = 1/n * np.sum(res * x_train)
    dc = 1/n * np.sum(res)
    gm = abs(dm) / (abs(m) + 1e-10)
    gc = abs(dc) / (abs(c) + 1e-10)
    rel_err = max(gm, gc)
    lr = r_target / (rel_err + 1e-10)
    lr = np.clip(lr, 1e-8, 0.05)

    m = m - lr * dm
    c = c - lr * dc
    
y_pred = m * x_test + c
df3 = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})
ss_res = ((y_test - y_pred)**2).sum()
ss_tot = ((y_test - y_test.mean())**2).sum()

r2 = 1 - ss_res/ss_tot
print(f"y = {m:.4f}x + {c:.4f}")
print(f"r2 score = {r2}")
print(f"epoch = {epoch}")
df3

y = 0.9979x + 0.0294
r2 score = 0.9566484453977263
epoch = 3218


,Actual,Predicted
2,-1.419919,-1.334441
17,0.262859,0.024627
1,-1.105527,-1.405971
9,-0.698013,-0.547612
11,-0.749769,-0.440317
5,-0.718307,-0.833732
18,0.198860,0.239216
24,1.240203,1.240635
27,1.359074,1.562519
29,1.701773,1.884404


In [12]:
# Moore Penrose pseudoinverse

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.4, random_state = 7, shuffle = True)

x = np.concat([np.ones((x_train.shape[0], 1)), x_train.values.reshape(-1, 1)], axis = 1)
y = y_train.values.reshape(-1,1)
eigval, eig_vectors = np.linalg.eig((x.T @ x))

singval = np.sqrt(eigval)
summation = np.diag(singval)
U = x @ eig_vectors @ np.linalg.inv(summation)
x_pinv = eig_vectors @ np.linalg.inv(summation) @ U.T
theta = x_pinv @ y
c, m = theta[0][0], theta[1][0]

y_pred = m * x_test + c
df5 = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})
ss_res = ((y_test - y_pred)**2).sum()
ss_tot = ((y_test - y_test.mean())**2).sum()

r2 = 1 - ss_res/ss_tot
print(f"y = {m:.4f}x + {c:.4f}")
print(f"r2 score = {r2}")
df5

y = 0.9979x + 0.0294
r2 score = 0.956648445382114


,Actual,Predicted
2,-1.419919,-1.334441
17,0.262859,0.024627
1,-1.105527,-1.405971
9,-0.698013,-0.547612
11,-0.749769,-0.440317
5,-0.718307,-0.833732
18,0.198860,0.239216
24,1.240203,1.240635
27,1.359074,1.562519
29,1.701773,1.884404


In [16]:
type(y)

numpy.ndarray